In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works when the notebook starts in the repository root or notebooks folder
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detectors.duplicate_detector import DuplicateDetector

print("Repository:", ROOT)


Repository: C:\Users\bobcl\IFN735 CAPSTONE


In [2]:
batch_file = ROOT / "data/raw/ResultSet.csv"
reference_file = ROOT / "data/reference/DUP_historical_reference_model.csv"

assert batch_file.is_file(), f"Missing: {batch_file}"
assert reference_file.is_file(), f"Missing: {reference_file}"

df = pd.read_csv(
    batch_file,
    dtype={"INSTRUMENT_ID": "string"}
)

print(f"Batch rows: {len(df):,}")


Batch rows: 99,999


In [3]:
detector = DuplicateDetector(repository_root=ROOT)

result = detector.detect(df)

print(result.details["reason"])
display(pd.DataFrame(
    [result.details["validation_report"]]
))


462 of 4437 assessed Duplicate observations flagged for chemist review


,n_input,n_duplicate,n_missing_analyte,n_prepared,n_eligible,n_ineligible,n_missing_reference,n_scored,n_invalid_q95_reference,n_q95_assessable,n_reference_negative_q95_points,n_outside_reference_range
0,99999,5075,136,4939,4437,502,0,4437,1,4429,41,7


In [4]:
scored_df = pd.DataFrame(result.details["results"])
ranked_df = pd.DataFrame(result.details["ranked_candidates"])

# Input eligibility and allowable-RPD assessment remain unchanged.
assert len(scored_df) == 4437
assert scored_df["EXCEEDS_ALLOWABLE_RPD"].sum() == 205

# Invalid Q95 thresholds must never trigger historical breaches.
invalid_q95 = ~scored_df["VALID_Q95_REFERENCE"]

assert not scored_df.loc[invalid_q95, "EXCEEDS_Q95"].any()
assert scored_df.loc[invalid_q95, "Q95_RATIO"].isna().all()

# Rankings contain actual breach candidates only.
assert ranked_df["MAX_ANOMALY_CLASS"].isin([
    "HISTORICAL_HIGH",
    "LIMIT_EXCEEDED",
    "BOTH_EXCEEDED"
]).all()

print(f"Q95 exceedances: {scored_df['EXCEEDS_Q95'].sum():,}")
print(f"Invalid Q95 observations: {invalid_q95.sum():,}")
print(f"Ranked candidates: {len(ranked_df):,}")
print("Defensive Q95 checks passed.")

display(ranked_df.head(10))


Q95 exceedances: 366
Invalid Q95 observations: 1
Ranked candidates: 377
Defensive Q95 checks passed.


,JOB_CODE,ANALYTE_CODE,ANOMALY_COUNT,MAX_CLASS_PRIORITY,MAX_RPD,MAX_Q95_RATIO,MAX_ALLOWABLE_RATIO,Q95_EXCEED_COUNT,ALLOWABLE_EXCEED_COUNT,BOTH_EXCEED_COUNT,TOTAL_OBSERVATION_COUNT,ANOMALY_RATE,MAX_ANOMALY_CLASS,RANK
0,TSV_LB0006483071,ZR,3,3,149.96,1.869498,9.556851,2,3,2,3,1.0,BOTH_EXCEEDED,1
1,TSV_LB0006483071,HF,3,3,66.01,1.028132,1.755075,2,3,2,3,1.0,BOTH_EXCEEDED,2
2,TSV_LB0004328823,NI,2,3,176.54,7.109358,8.809697,2,2,2,4,0.5,BOTH_EXCEEDED,3
3,TSV_LB0004328823,FE,2,3,182.30,1.980588,6.329937,2,2,2,4,0.5,BOTH_EXCEEDED,4
4,TSV_LB0004797270,CO,2,3,180.96,3.581341,5.069469,2,2,2,4,0.5,BOTH_EXCEEDED,5
5,TSV_LB0007353672,CR,1,3,55.78,1.000032,3.120331,1,1,1,1,1.0,BOTH_EXCEEDED,6
6,TSV_LB0007353672,BA,1,3,47.81,4.332689,3.092176,1,1,1,1,1.0,BOTH_EXCEEDED,7
7,TSV_LB0007353672,S,1,3,51.15,1.000001,2.630846,1,1,1,1,1.0,BOTH_EXCEEDED,8
8,TSV_LB0006832499,AS,3,3,1403.40,1.707586,2.514617,1,3,1,3,1.0,BOTH_EXCEEDED,9
9,TSV_LB0007353672,V,1,3,49.53,1.825327,2.445609,1,1,1,1,1.0,BOTH_EXCEEDED,10


In [5]:
display(
    scored_df.loc[
        (scored_df["JOB_CODE"] == "TSV_LB0004797270")
        & (scored_df["ANALYTE_CODE"] == "AS"),
        [
            "MEAN_CONC", "RPD", "REF_Q95",
            "WITHIN_REFERENCE_RANGE",
            "EXCEEDS_Q95", "Q95_RATIO", "ANOMALY_CLASS"
        ]
    ]
)


,MEAN_CONC,RPD,REF_Q95,WITHIN_REFERENCE_RANGE,EXCEEDS_Q95,Q95_RATIO,ANOMALY_CLASS
291,117.48,204.75,-38.732844,True,False,NaN,LIMIT_EXCEEDED


In [6]:
display(pd.DataFrame([result.details["validation_report"]]))

display(pd.DataFrame(
    result.details["invalid_q95_observations"]
))


,n_input,n_duplicate,n_missing_analyte,n_prepared,n_eligible,n_ineligible,n_missing_reference,n_scored,n_invalid_q95_reference,n_q95_assessable,n_reference_negative_q95_points,n_outside_reference_range
0,99999,5075,136,4939,4437,502,0,4437,1,4429,41,7


,JOB_CODE,ANALYSED_DATE,ANALYTE_CODE,SCHEME_CODE,PARENT_NUMERIC_FINAL_VALUE,NUMERIC_FINAL_VALUE,STAT_DL_DUP_VALUE,LIM_REP_DUP_VALUE,PRECISION_STATUS,UNIT_CODE,...,VALID_Q95_REFERENCE,Q95_ASSESSABLE,EXCEEDS_Q95,EXCEEDS_ALLOWABLE_RPD,Q95_EXCESS,ALLOWABLE_EXCESS,Q95_RATIO,ALLOWABLE_RATIO,ANOMALY_CLASS,CLASS_PRIORITY
0,TSV_LB0004797270,2020-11-24 12:04:28.000000,AS,GE_ICP40Q12,237.75,-2.79,7.5,15.0,Failure,MG_KG,...,False,False,False,True,None,183.364954,None,9.574448,LIMIT_EXCEEDED,2


In [8]:
from pathlib import Path
from src.visualisers.duplicate_visualize import generate_duplicate_plots

plot_dir = ROOT / "outputs" / "duplicate_plots"

plots = generate_duplicate_plots(
    result=result,
    output_dir=plot_dir,
    top_n=5,
)

expected = result.details["ranked_candidates"][:5]
expected_keys = {
    (row["JOB_CODE"], row["ANALYTE_CODE"])
    for row in expected
}

assert set(plots) == expected_keys
assert all(isinstance(path, Path) for path in plots.values())
assert all(path.exists() and path.stat().st_size > 0 for path in plots.values())

print(f"Passed: {len(plots)} charts generated for the expected candidates.")


Passed: 5 charts generated for the expected candidates.
